In [20]:
import numpy as np
from math import sqrt, atan2, pi

np.set_printoptions(
    precision=6,
    suppress=True,
    linewidth=160
)

# ============================================================
# 1. INPUT
# ============================================================
# Kolonnerekkefølge i ukjente:
# [orientering_T620, orientering_T827, dX_T620, dY_T620, dX_T827, dY_T827]
#
# Skalere til annen oppgave:
# - legg inn kjente og ukjente punkter i punkter
# - legg ukjente punkter inn i ukjente
# - legg stasjoner med orienteringsukjent inn i orienteringer
# - oppdater retning_obs og avstand_obs

punkter = {
    # "Punktnavn": [X, Y, kjent?]

    "A": [-885.942,   39.263,  True],
    "B": [0.000,      0.000,  True],
    "C": [-188.967, 1331.959, True],
    "D": [-819.168, 1315.756, True],

    "1": [-236.962,  677.988, False],
    "2": [-437.931,  665.832, False],
    "3": [-624.567,  667.941, False],
}

# orienteringsukjente første kolonne 0 og andre i A matrisen
orienteringer = {
    "A": 0,
    "B": 1,
    "C": 2,
    "D": 3,
}

# koordinatukjente
ukjente = {
    "1": [4, 5],
    "2": [6, 7],
    "3": [8, 9],
}

antall_ukjente = 10 #O_A, O_B,O_C,O_D,1x_x,1x_y,2x_x,2x_y,3x_x,3x_y OGSA 32-10= 22 overbestemmelser obs-ukjente

retning_obs = [
    # [fra, til, observert retning i gon]

    ["A", "B", 0.0000],
    ["A", "1", 52.3140],
    ["A", "2", 63.3012],
    ["A", "3", 77.7350],
    ["A", "D", 99.4937],

    ["B", "C", 0.0000],
    ["B", "1", 12.4318],
    ["B", "2", 28.0667],
    ["B", "3", 38.8933],
    ["B", "A", 88.2074],

    ["C", "D", 0.0000],
    ["C", "3", 61.4030],
    ["C", "2", 75.5923],
    ["C", "1", 93.7009],
    ["C", "B", 107.3340],

    ["D", "A", 0.0000],
    ["D", "3", 21.9039],
    ["D", "2", 37.1011],
    ["D", "1", 50.4292],
    ["D", "C", 104.9644],
]

avstand_obs = [
    # [fra, til, observert avstand i meter]

    ["A", "1", 910.582],
    ["A", "2", 770.264],
    ["A", "3", 680.854],

    ["B", "1", 718.213],
    ["B", "2", 796.945],
    ["B", "3", 914.463],

    ["C", "3", 794.154],
    ["C", "2", 711.135],
    ["C", "1", 655.737],

    ["D", "3", 676.420],
    ["D", "2", 753.490],
    ["D", "1", 863.553],
]

sigma_retning = 0.0005  # gon


def sigma_avstand(D):
    # D i meter
    return 0.003 + 0.003 * (D / 1000)


# ============================================================
# 2. HJELPEFUNKSJONER
# ============================================================

def normaliser_gon(v):
    # holder vinkler i området -200 til 200 gon
    while v > 200:
        v -= 400
    while v < -200:
        v += 400
    return v


def beregn_retning(fra_punkt, til_punkt):
    XA, YA = punkter[fra_punkt][0], punkter[fra_punkt][1]
    XB, YB = punkter[til_punkt][0], punkter[til_punkt][1]

    dX = XB - XA
    dY = YB - YA

    r = atan2(dY, dX) * 200 / pi

    if r < 0:
        r += 400

    return r


def beregn_avstand(fra_punkt, til_punkt):
    XA, YA = punkter[fra_punkt][0], punkter[fra_punkt][1]
    XB, YB = punkter[til_punkt][0], punkter[til_punkt][1]

    dX = XB - XA
    dY = YB - YA

    return sqrt(dX**2 + dY**2)


# ============================================================
# 3. STARTORIENTERINGER
# ============================================================
# Første retning fra hver stasjon brukes som referanse.
# Dette gir l = 0 for første retning fra hver stasjon.

orientering_start = {}

for fra_punkt in orienteringer:
    for obs in retning_obs:
        if obs[0] == fra_punkt:
            til_punkt = obs[1]
            obs_retning = obs[2]

            beregnet = beregn_retning(fra_punkt, til_punkt)

            orientering_start[fra_punkt] = normaliser_gon(
                beregnet - obs_retning
            )
            break


# ============================================================
# 4. OBSERVASJONSLIKNINGER
# ============================================================

def lag_retning_obs(fra_punkt, til_punkt, obs_retning):
    XA, YA = punkter[fra_punkt][0], punkter[fra_punkt][1]
    XB, YB = punkter[til_punkt][0], punkter[til_punkt][1]

    dX = XB - XA
    dY = YB - YA
    D = sqrt(dX**2 + dY**2)

    beregnet = beregn_retning(fra_punkt, til_punkt)
    orientering = orientering_start[fra_punkt]

    # Rapport-vennlig konvensjon:
    # l = obs + orientering - beregnet
    l = normaliser_gon(obs_retning + orientering - beregnet)

    A_rad = [0] * antall_ukjente

    # Viktig: -1 for å følge rapportens A-matrise
    A_rad[orienteringer[fra_punkt]] = -1

    # derivert for til-punkt
    if til_punkt in ukjente:
        colX, colY = ukjente[til_punkt]

        A_rad[colX] = -(dY / D**2) * 200 / pi
        A_rad[colY] =  (dX / D**2) * 200 / pi

    # derivert for fra-punkt
    if fra_punkt in ukjente:
        colX, colY = ukjente[fra_punkt]

        A_rad[colX] =  (dY / D**2) * 200 / pi
        A_rad[colY] = -(dX / D**2) * 200 / pi

    # relativ vekt, slik rapporten bruker
    p = 1

    return A_rad, l, p


def lag_avstand_obs(fra_punkt, til_punkt, obs_avstand):
    XA, YA = punkter[fra_punkt][0], punkter[fra_punkt][1]
    XB, YB = punkter[til_punkt][0], punkter[til_punkt][1]

    dX = XB - XA
    dY = YB - YA
    D = sqrt(dX**2 + dY**2)

    # l = observert - beregnet
    l = obs_avstand - D

    A_rad = [0] * antall_ukjente

    # derivert for til-punkt
    if til_punkt in ukjente:
        colX, colY = ukjente[til_punkt]

        A_rad[colX] = dX / D
        A_rad[colY] = dY / D

    # derivert for fra-punkt
    if fra_punkt in ukjente:
        colX, colY = ukjente[fra_punkt]

        A_rad[colX] = -dX / D
        A_rad[colY] = -dY / D

    sigma = sigma_avstand(D)

    # relativ vekt mot retning
    p = sigma_retning**2 / sigma**2

    return A_rad, l, p


# ============================================================
# 5. BYGG A, l OG P
# ============================================================

A_liste = []
l_liste = []
p_liste = []
navn_liste = []

for fra_punkt, til_punkt, obs_retning in retning_obs:
    A_rad, l_verdi, p_verdi = lag_retning_obs(
        fra_punkt,
        til_punkt,
        obs_retning
    )

    A_liste.append(A_rad)
    l_liste.append(l_verdi)
    p_liste.append(p_verdi)
    navn_liste.append(f"Retning {fra_punkt}->{til_punkt}")

for fra_punkt, til_punkt, obs_avstand in avstand_obs:
    A_rad, l_verdi, p_verdi = lag_avstand_obs(
        fra_punkt,
        til_punkt,
        obs_avstand
    )

    A_liste.append(A_rad)
    l_liste.append(l_verdi)
    p_liste.append(p_verdi)
    navn_liste.append(f"Avstand {fra_punkt}->{til_punkt}")

A = np.array(A_liste, dtype=float)
l = np.array(l_liste, dtype=float).reshape(-1, 1)
P = np.diag(p_liste)


# ============================================================
# 6. MINSTE KVADRATERS METODE
# ============================================================

N = A.T @ P @ A
h = A.T @ P @ l

dx = np.linalg.solve(N, h)

v = A @ dx - l

frihetsgrader = len(l) - antall_ukjente

s0_hat = sqrt((v.T @ P @ v)[0, 0] / frihetsgrader)

Qxx = np.linalg.inv(N)

std_parametre = s0_hat * np.sqrt(np.diag(Qxx)).reshape(-1, 1)


# ============================================================
# 7. PEN UTSKRIFT
# ============================================================

kolonnenavn = [
    "o_A",
    "o_B",
    "o_C",
    "o_D",

    "dX_1",
    "dY_1",

    "dX_2",
    "dY_2",

    "dX_3",
    "dY_3",
] #NB ENDRE PÅ EXAMENNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNN

def print_matrix(navn, M, rader=None, kolonner=None, des=6):
    print("\n" + "=" * 80)
    print(navn)
    print("=" * 80)

    M = np.array(M)

    if M.ndim == 1:
        M = M.reshape(-1, 1)

    if kolonner is None:
        kolonner = [f"c{i+1}" for i in range(M.shape[1])]

    if rader is None:
        rader = [f"r{i+1}" for i in range(M.shape[0])]

    print("".ljust(24), end="")
    for k in kolonner:
        print(f"{k:>14}", end="")
    print()

    for navn_rad, row in zip(rader, M):
        print(f"{navn_rad:<24}", end="")
        for verdi in row:
            print(f"{verdi:>14.{des}f}", end="")
        print()


print("\nSTARTORIENTERINGER")
for punkt, verdi in orientering_start.items():
    print(f"{punkt}: {verdi:.6f} gon")

print_matrix("A-matrise", A, navn_liste, kolonnenavn, des=6)
print_matrix("l-vektor", l, navn_liste, ["l"], des=6)
print_matrix("P diagonal", np.diag(P), navn_liste, ["p"], des=8)

print_matrix("N = A.T P A", N, kolonnenavn, kolonnenavn, des=6)
print_matrix("h = A.T P l", h, kolonnenavn, ["h"], des=8)

print_matrix("Korreksjoner dx", dx, kolonnenavn, ["dx"], des=8)
print_matrix("Residualer v = A dx - l", v, navn_liste, ["v"], des=8)

print("\n" + "=" * 80)
print("STANDARDAVVIK TIL VEKTSENHETEN")
print("=" * 80)
print(f"s0_hat = {s0_hat:.8f}")

print_matrix("Qxx = N^-1", Qxx, kolonnenavn, kolonnenavn, des=6)
print_matrix("Standardavvik til parametere", std_parametre, kolonnenavn, ["std"], des=8)


print("\n" + "=" * 80)
print("NYE KOORDINATER")
print("=" * 80)

for punkt in ukjente:
    colX, colY = ukjente[punkt]

    X_gammel = punkter[punkt][0]
    Y_gammel = punkter[punkt][1]

    X_ny = X_gammel + dx[colX, 0]
    Y_ny = Y_gammel + dx[colY, 0]

    print(f"{punkt}:")
    print(f"  X gammel = {X_gammel:.6f}")
    print(f"  Y gammel = {Y_gammel:.6f}")
    print(f"  dX       = {dx[colX, 0]:.8f}")
    print(f"  dY       = {dx[colY, 0]:.8f}")
    print(f"  X ny     = {X_ny:.6f}")
    print(f"  Y ny     = {Y_ny:.6f}")


print("\n" + "=" * 80)
print("ORIENTERINGSKORREKSJONER")
print("=" * 80)

for punkt in orienteringer:
    colO = orienteringer[punkt]

    print(f"{punkt}: do = {dx[colO, 0]:.8f} gon")


STARTORIENTERINGER
A: -2.819514 gon
B: 108.971945 gon
C: -198.363557 gon
D: -103.327158 gon

A-matrise
                                   o_A           o_B           o_C           o_D          dX_1          dY_1          dX_2          dY_2          dX_3          dY_3
Retning A->B                 -1.000000      0.000000      0.000000      0.000000      0.000000      0.000000      0.000000      0.000000      0.000000      0.000000
Retning A->1                 -1.000000      0.000000      0.000000      0.000000     -0.049041      0.049829      0.000000      0.000000      0.000000      0.000000
Retning A->2                 -1.000000      0.000000      0.000000      0.000000      0.000000      0.000000     -0.067231      0.048072      0.000000      0.000000
Retning A->3                 -1.000000      0.000000      0.000000      0.000000      0.000000      0.000000      0.000000      0.000000     -0.086339      0.035896
Retning A->D                 -1.000000      0.000000      0.000000     

In [22]:
from scipy.stats import chi2

# ============================================================
# 8. TEST AV STANDARDAVVIK TIL VEKTSENHETEN
# ============================================================

# antall observasjoner og ukjente
n = len(l)
u = antall_ukjente
f = n - u  # frihetsgrader

# apriori standardavvik til vektsenheten
# siden vi bruker relative vekter med retning som referanse
sigma0 = sigma_retning

# estimert standardavvik til vektsenheten
sigma0_hat = sqrt((v.T @ P @ v)[0, 0] / f)

# teststørrelse
chi2_test = (f * sigma0_hat**2) / sigma0**2

# signifikansnivå
alpha = 0.05

# kritiske grenser, tosidig test
chi2_nedre = chi2.ppf(alpha / 2, f)
chi2_ovre = chi2.ppf(1 - alpha / 2, f)

print("\n" + "=" * 80)
print("TEST AV STANDARDAVVIK TIL VEKTSENHETEN")
print("=" * 80)

print(f"Antall observasjoner n        = {n}")
print(f"Antall ukjente u              = {u}")
print(f"Frihetsgrader f = n - u       = {f}")

print(f"\nApriori sigma0                = {sigma0:.8f}")
print(f"Estimert sigma0_hat           = {sigma0_hat:.8f}")

print(f"\nChi2 teststørrelse            = {chi2_test:.8f}")
print(f"Chi2 nedre grense             = {chi2_nedre:.8f}")
print(f"Chi2 øvre grense              = {chi2_ovre:.8f}")

if chi2_nedre <= chi2_test <= chi2_ovre:
    print("\nKonklusjon:")
    print("Beholder H0.")
    print("Estimert standardavvik er ikke signifikant forskjellig fra apriori verdi.\nSiden estimert standardavvik til vektsenheten ikke er signifikant forskjellig \nfra a priori-verdien, tyder dette på at observasjonene og vektmodellen er konsistente.\nUtjevningen virker derfor stabil og pålitelig.")
else:
    print("\nKonklusjon:")
    print("Forkaster H0.")
    print("Estimert standardavvik er signifikant forskjellig fra apriori verdi.")


TEST AV STANDARDAVVIK TIL VEKTSENHETEN
Antall observasjoner n        = 32
Antall ukjente u              = 10
Frihetsgrader f = n - u       = 22

Apriori sigma0                = 0.00050000
Estimert sigma0_hat           = 0.00100092

Chi2 teststørrelse            = 88.16116826
Chi2 nedre grense             = 10.98232073
Chi2 øvre grense              = 36.78071208

Konklusjon:
Forkaster H0.
Estimert standardavvik er signifikant forskjellig fra apriori verdi.


## Multippel T-test

In [24]:
# ============================================================
# 9. OBSERVASJONSTEST / MULTIPPEL T-TEST FOR FLERE OBSERVASJONER
# ============================================================

from scipy.stats import t
from math import sqrt
import numpy as np

observasjoner_som_testes = [
    "Retning A->B",
    "Retning A->1",
    "Avstand A->1",
]

alpha = 0.05
n = len(l)
u = antall_ukjente
f = n - u

Qvv = np.linalg.inv(P) - A @ Qxx @ A.T

alpha_i = alpha / n
t_kritisk = t.ppf(1 - alpha_i / 2, f)

print("\n" + "=" * 80)
print("OBSERVASJONSTEST / MULTIPPEL T-TEST")
print("=" * 80)

print(f"Antall observasjoner n       = {n}")
print(f"Antall ukjente u             = {u}")
print(f"Frihetsgrader f              = {f}")
print(f"Signifikansnivå alpha        = {alpha}")
print(f"Bonferroni alpha_i           = {alpha_i:.8f}")
print(f"Kritisk t-verdi              = {t_kritisk:.8f}")

for observasjon_som_testes in observasjoner_som_testes:

    if observasjon_som_testes not in navn_liste:
        print("\n" + "-" * 80)
        print(f"Fant ikke observasjon: {observasjon_som_testes}")
        print("Sjekk stavemåten mot navn_liste.")
        continue

    i = navn_liste.index(observasjon_som_testes)

    vi = v[i, 0]
    Qvv_i = Qvv[i, i]
    s_vi = s0_hat * sqrt(Qvv_i)
    ti = abs(vi) / s_vi

    print("\n" + "-" * 80)
    print(f"Observasjon som testes        = {observasjon_som_testes}")
    print(f"Radnummer i A/l/v             = {i + 1}")

    print("\n--- Residual og usikkerhet ---")
    print(f"Residual v_i                  = {vi:.8f}")
    print(f"Qvv_i                         = {Qvv_i:.8f}")
    print(f"Standardavvik s_vi            = {s_vi:.8f}")

    print("\n--- Test ---")
    print(f"Teststørrelse |t_i|           = {ti:.8f}")
    print(f"Kritisk t-verdi               = {t_kritisk:.8f}")

    print("\n--- Konklusjon ---")
    if ti > t_kritisk:
        print("Mulig grov feil: |t_i| er større enn kritisk verdi.")
    else:
        print("Ingen signifikant grov feil: |t_i| er mindre enn kritisk verdi.")


OBSERVASJONSTEST / MULTIPPEL T-TEST
Antall observasjoner n       = 32
Antall ukjente u             = 10
Frihetsgrader f              = 22
Signifikansnivå alpha        = 0.05
Bonferroni alpha_i           = 0.00156250
Kritisk t-verdi              = 3.60762463

--------------------------------------------------------------------------------
Observasjon som testes        = Retning A->B
Radnummer i A/l/v             = 1

--- Residual og usikkerhet ---
Residual v_i                  = 0.00032659
Qvv_i                         = 0.76116052
Standardavvik s_vi            = 0.00087324

--- Test ---
Teststørrelse |t_i|           = 0.37399702
Kritisk t-verdi               = 3.60762463

--- Konklusjon ---
Ingen signifikant grov feil: |t_i| er mindre enn kritisk verdi.

--------------------------------------------------------------------------------
Observasjon som testes        = Retning A->1
Radnummer i A/l/v             = 2

--- Residual og usikkerhet ---
Residual v_i                  = -0.0014413

## En tilleggsparameter for målestokk

In [25]:
# ============================================================
# DEL C: TEST AV MÅLESTOKKSPARAMETER
# ============================================================

from scipy.stats import t

# Legg til én ekstra ukjent: målestokk m
kol_m = antall_ukjente
antall_ukjente = antall_ukjente + 1

# Viktig: bygg kolonnenavn på nytt#############################################################
kolonnenavn = [
    "o_A",
    "o_B",
    "o_C",
    "o_D",

    "dX_1",
    "dY_1",

    "dX_2",
    "dY_2",

    "dX_3",
    "dY_3",
]


def lag_retning_obs_med_m(fra_punkt, til_punkt, obs_retning):
    XA, YA = punkter[fra_punkt][0], punkter[fra_punkt][1]
    XB, YB = punkter[til_punkt][0], punkter[til_punkt][1]

    dX = XB - XA
    dY = YB - YA
    D = sqrt(dX**2 + dY**2)

    beregnet = beregn_retning(fra_punkt, til_punkt)
    orientering = orientering_start[fra_punkt]

    l = normaliser_gon(obs_retning + orientering - beregnet)

    A_rad = [0] * antall_ukjente

    # orientering
    A_rad[orienteringer[fra_punkt]] = -1

    # til-punkt
    if til_punkt in ukjente:
        colX, colY = ukjente[til_punkt]
        A_rad[colX] = -(dY / D**2) * 200 / pi
        A_rad[colY] =  (dX / D**2) * 200 / pi

    # fra-punkt
    if fra_punkt in ukjente:
        colX, colY = ukjente[fra_punkt]
        A_rad[colX] =  (dY / D**2) * 200 / pi
        A_rad[colY] = -(dX / D**2) * 200 / pi

    # målestokk påvirker ikke retninger
    A_rad[kol_m] = 0

    p = 1

    return A_rad, l, p


def lag_avstand_obs_med_m(fra_punkt, til_punkt, obs_avstand):
    XA, YA = punkter[fra_punkt][0], punkter[fra_punkt][1]
    XB, YB = punkter[til_punkt][0], punkter[til_punkt][1]

    dX = XB - XA
    dY = YB - YA
    D = sqrt(dX**2 + dY**2)

    l = obs_avstand - D

    A_rad = [0] * antall_ukjente

    # til-punkt
    if til_punkt in ukjente:
        colX, colY = ukjente[til_punkt]
        A_rad[colX] = dX / D
        A_rad[colY] = dY / D

    # fra-punkt
    if fra_punkt in ukjente:
        colX, colY = ukjente[fra_punkt]
        A_rad[colX] = -dX / D
        A_rad[colY] = -dY / D

    # målestokksparameter:
    # D_observert = D_beregnet + m * D_beregnet
    # derfor derivert med hensyn på m = D
    A_rad[kol_m] = D

    sigma = sigma_avstand(D)
    p = sigma_retning**2 / sigma**2

    return A_rad, l, p


# ============================================================
# BYGG A, l OG P PÅ NYTT
# ============================================================

A_liste = []
l_liste = []
p_liste = []
navn_liste = []

for fra_punkt, til_punkt, obs_retning in retning_obs:
    A_rad, l_verdi, p_verdi = lag_retning_obs_med_m(
        fra_punkt,
        til_punkt,
        obs_retning
    )

    A_liste.append(A_rad)
    l_liste.append(l_verdi)
    p_liste.append(p_verdi)
    navn_liste.append(f"Retning {fra_punkt}->{til_punkt}")

for fra_punkt, til_punkt, obs_avstand in avstand_obs:
    A_rad, l_verdi, p_verdi = lag_avstand_obs_med_m(
        fra_punkt,
        til_punkt,
        obs_avstand
    )

    A_liste.append(A_rad)
    l_liste.append(l_verdi)
    p_liste.append(p_verdi)
    navn_liste.append(f"Avstand {fra_punkt}->{til_punkt}")

A = np.array(A_liste, dtype=float)
l = np.array(l_liste, dtype=float).reshape(-1, 1)
P = np.diag(p_liste)


# ============================================================
# NY UTJEVNING
# ============================================================

N = A.T @ P @ A
h = A.T @ P @ l

dx = np.linalg.solve(N, h)

v = A @ dx - l

n = len(l)
u = antall_ukjente
f = n - u

s0_hat = sqrt((v.T @ P @ v)[0, 0] / f)

Qxx = np.linalg.inv(N)

std_parametre = s0_hat * np.sqrt(np.diag(Qxx)).reshape(-1, 1)


# ============================================================
# MÅLESTOKK I PPM
# ============================================================

m = dx[kol_m, 0]
m_ppm = m * 1_000_000

std_m = std_parametre[kol_m, 0]
std_m_ppm = std_m * 1_000_000

t_m = abs(m / std_m)

alpha = 0.05
t_kritisk = t.ppf(1 - alpha / 2, f)


print("\n" + "=" * 80)
print("DEL C: UTJEVNING MED MÅLESTOKKSPARAMETER")
print("=" * 80)

print("\nMålestokksparameter:")
print(f"m              = {m:.12f}")
print(f"m i ppm        = {m_ppm:.4f} ppm")

print("\nStandardavvik:")
print(f"std(m)         = {std_m:.12f}")
print(f"std(m) i ppm   = {std_m_ppm:.4f} ppm")

print("\nTest av målestokksparameter:")
print(f"t-verdi        = {t_m:.6f}")
print(f"kritisk t      = {t_kritisk:.6f}")
print(f"frihetsgrader  = {f}")

print("\nKort kommentar:")

if t_m > t_kritisk:
    print(
        "Målestokksparameteren er signifikant forskjellig fra null. "
        "Dette tyder på at avstandene har en målestokksfeil."
    )
else:
    print(
        "Målestokksparameteren er ikke signifikant forskjellig fra null. "
        "Det er derfor ikke grunnlag for å si at avstandene har en målestokksfeil."
    )


print("\nNye koordinater:")

for punkt in ukjente:
    colX, colY = ukjente[punkt]

    X_ny = punkter[punkt][0] + dx[colX, 0]
    Y_ny = punkter[punkt][1] + dx[colY, 0]

    print(f"{punkt}: X = {X_ny:.6f}, Y = {Y_ny:.6f}")


DEL C: UTJEVNING MED MÅLESTOKKSPARAMETER

Målestokksparameter:
m              = 0.000007074329
m i ppm        = 7.0743 ppm

Standardavvik:
std(m)         = 0.000003863053
std(m) i ppm   = 3.8631 ppm

Test av målestokksparameter:
t-verdi        = 1.831279
kritisk t      = 2.079614
frihetsgrader  = 21

Kort kommentar:
Målestokksparameteren er ikke signifikant forskjellig fra null. Det er derfor ikke grunnlag for å si at avstandene har en målestokksfeil.

Nye koordinater:
1: X = -236.955556, Y = 677.990173
2: X = -437.930182, Y = 665.830877
3: X = -624.566263, Y = 667.935509


## Indre/Ytre politlighet

In [26]:
# ============================================================
# 10. INDRE OG YTRE PÅLITELIGHET
# ============================================================
#
# Forutsetter at du allerede har:
# A, P, Qxx, navn_liste, s0_hat
#
# A    = designmatrise
# P    = vektsmatrise
# Qxx  = inv(N)
# navn_liste = observasjonsnavn
# s0_hat = estimert standardavvik til vektsenheten

import numpy as np
import pandas as pd
from scipy.stats import norm

# ------------------------------------------------------------
# Grunnverdier
# ------------------------------------------------------------

alpha = 0.05
beta = 0.20          # vanlig valg: teststyrke 80 %, altså beta = 20 %
n = A.shape[0]
u = A.shape[1]

# ------------------------------------------------------------
# Kofaktormatrise for residualer
# ------------------------------------------------------------

Qll = np.linalg.inv(P)
Qvv = Qll - A @ Qxx @ A.T

# ------------------------------------------------------------
# Redundanstall
# r_i = p_i * q_vv_i
# ------------------------------------------------------------

p_diag = np.diag(P)
qvv_diag = np.diag(Qvv)

redundans = p_diag * qvv_diag

# ------------------------------------------------------------
# Indre pålitelighet
# Minimal påvisbar grov feil
#
# nabla_i = delta0 * sigma0 / sqrt(p_i * r_i)
#
# delta0 finnes ofte fra alpha og beta.
# Enkel eksamensvennlig normaltilnærming:
# delta0 = z_(1-alpha/2) + z_(1-beta)
# ------------------------------------------------------------

z_alpha = norm.ppf(1 - alpha / 2)
z_beta = norm.ppf(1 - beta)

delta0 = z_alpha + z_beta

indre = delta0 * s0_hat / np.sqrt(p_diag * redundans)

# ------------------------------------------------------------
# Ytre pålitelighet
#
# Effekt på parameterne dersom observasjon i har grov feil nabla_i:
#
# dx_grov = Qxx A.T P e_i nabla_i
#
# Siden e_i bare velger observasjon i:
# dx_grov = Qxx @ A[i,:].T * p_i * nabla_i
# ------------------------------------------------------------

ytre_liste = []

for i in range(n):
    ai = A[i, :].reshape(-1, 1)
    dx_effekt = Qxx @ ai * p_diag[i] * indre[i]

    ytre_liste.append(dx_effekt.flatten())

ytre = np.array(ytre_liste)

# ------------------------------------------------------------
# Lag tabell
# ------------------------------------------------------------

resultat = pd.DataFrame({
    "observasjon": navn_liste,
    "p_i": p_diag,
    "qvv_i": qvv_diag,
    "redundans r_i": redundans,
    "indre_palitelighet": indre,
})

# Legg til ytre effekt for hver parameter
for j, kol in enumerate(kolonnenavn):
    resultat[f"ytre_{kol}"] = ytre[:, j]

# Total ytre effekt på koordinater, hvis du vil ha én enkel størrelse
# Tilpass kolonnenavn hvis du bruker andre navn.
koord_kolonner = [k for k in kolonnenavn if k.startswith("dX") or k.startswith("dY")]

for obs_i in range(n):
    pass

ytre_koord_norm = []

for i in range(n):
    verdier = []
    for j, kol in enumerate(kolonnenavn):
        if kol.startswith("dX") or kol.startswith("dY"):
            verdier.append(ytre[i, j])
    ytre_koord_norm.append(np.sqrt(np.sum(np.array(verdier) ** 2)))

resultat["ytre_koord_norm"] = ytre_koord_norm

# ------------------------------------------------------------
# Utskrift
# ------------------------------------------------------------

print("\n" + "=" * 100)
print("INDRE OG YTRE PÅLITELIGHET")
print("=" * 100)

print(f"alpha = {alpha}")
print(f"beta  = {beta}")
print(f"delta0 = {delta0:.4f}")
print(f"s0_hat = {s0_hat:.8f}")

print("\nKort oversikt:")
print(resultat[[
    "observasjon",
    "redundans r_i",
    "indre_palitelighet",
    "ytre_koord_norm"
]].to_string(index=False))

print("\nDårligst indre pålitelighet:")
print(resultat.loc[resultat["indre_palitelighet"].idxmax(), [
    "observasjon",
    "redundans r_i",
    "indre_palitelighet"
]])

print("\nStørst ytre effekt på koordinater:")
print(resultat.loc[resultat["ytre_koord_norm"].idxmax(), [
    "observasjon",
    "ytre_koord_norm"
]])


INDRE OG YTRE PÅLITELIGHET
alpha = 0.05
beta  = 0.2
delta0 = 2.8016
s0_hat = 0.00095132

Kort oversikt:
 observasjon  redundans r_i  indre_palitelighet  ytre_koord_norm
Retning A->B       0.761063            0.003055         0.004268
Retning A->1       0.678270            0.003236         0.006450
Retning A->2       0.638938            0.003334         0.007775
Retning A->3       0.587609            0.003477         0.009685
Retning A->D       0.761063            0.003055         0.004268
Retning B->C       0.761537            0.003054         0.004337
Retning B->1       0.597138            0.003449         0.009671
Retning B->2       0.645379            0.003318         0.007680
Retning B->3       0.680781            0.003230         0.006371
Retning B->A       0.761537            0.003054         0.004337
Retning C->D       0.749448            0.003079         0.005098
Retning C->3       0.641929            0.003326         0.007620
Retning C->2       0.604650            0.003428   

A-matrisa får én ekstra kolonne for målestokksparameteren.
Avstander får ikke-null elementer i denne kolonnen, mens retninger normalt får 0.
l-vektoren endres normalt ikke.
P-matrisa endres normalt ikke siden observasjonsnøyaktigheten er uendret.

## 4
For å teste for eventuell tvang i grunnlaget ville jeg sammenlignet en fri utjevning med en fast utjevning. I den frie utjevningen holdes ikke fastmerkene helt faste, mens i den faste utjevningen forutsettes koordinatene til A, B og C som feilfrie.

Deretter kan man undersøke om residualene eller estimert standardavvik til vektsenheten øker tydelig når grunnlaget holdes fast. Hvis den faste utjevningen gir betydelig dårligere tilpasning, kan det tyde på tvang eller feil i grunnlagspunktene.

Man kan også teste koordinatene til fastmerkene som observasjoner med egne vekter, i stedet for å behandle dem som helt feilfrie.

1. Grunnlagstest

Tester om fastpunktene inneholder tvang eller grove feil.

2. Observasjonstest

Tester om observasjoner inneholder grove feil ved hjelp av residualer og statistiske tester.

3. Indre pålitelighet

Beskriver hvor godt observasjonene kan oppdage grove feil.

4. Ytre pålitelighet

Viser hvor mye en eventuell uoppdaget grov feil påvirker de estimerte koordinatene.

5. Beregning av endelige koordinater

Når analysen er godkjent utføres endelig utjevning og koordinatene beregnes.

Test av standardavvik til vektsenheten

Hvis testen gir signifikant utslag kan det tyde på:

feil stokastisk modell,
grove feil i observasjonene,
eller tvang i grunnlaget.

Hvis testen ikke er signifikant stemmer observasjonene rimelig godt med den antatte modellen.

Hvor brukes fri-utjevning?

Fri-/tvangsfri utjevning brukes hovedsakelig i:

grunnlagstest
observasjonstest

for å unngå at feil skjules av faste punkter.

Manuell vs automatisk fri-utjevning
Manuell metode

Man innfører minimumsbetingelser eller fjerner nødvendige parametere manuelt.

Automatisk metode

Programvaren utfører fri-utjevning automatisk ved hjelp av pseudoinvers eller indre betingelser.

Indre vs ytre pålitelighet
Indre pålitelighet

Hvor godt systemet kan oppdage grove feil.

Ytre pålitelighet

Hvor stor effekt uoppdagede grove feil får på resultatene.

In [30]:
# ============================================================
# GMPE240 2024 - OPPGAVE 2: KALMANFILTER
# ============================================================

import sympy as sp
from IPython.display import display

# Symboler
Delta_t, q = sp.symbols("Delta_t q", positive=True)

# ============================================================
# OPPGAVE 2a - DYNAMISK MODELL
# ============================================================
#
# Dette er en konstant hastighet-modell.
# Kalman filter essens: x_dot = F x + G u ##############################
# 1D tilstand:
# [posisjon, hastighet]^T
#
# 3D tilstand:
# [X, X_dot, Y, Y_dot, Z, Z_dot]^T
#
# F beskriver dynamikken mellom posisjon og hastighet.
# G og u beskriver prosesstøy / tilfeldig akselerasjon.

F_1D = sp.Matrix([
    [0, 1],
    [0, 0]
])

G_1D = sp.Matrix([
    [0],
    [sp.sqrt(q)]
])

# ============================================================
# OPPGAVE 2b - PHI OG Q
# ============================================================
#
# Phi:
# ny posisjon = gammel posisjon + hastighet * Delta_t
# ny hastighet = gammel hastighet

Phi_1D = sp.Matrix([
    [1, Delta_t],
    [0, 1]
])

# Q:
# Prosesstøyens kovariansmatrise for konstant hastighet-modell

Q_1D = q * sp.Matrix([
    [Delta_t**3 / 3, Delta_t**2 / 2],
    [Delta_t**2 / 2, Delta_t]
])

# 3D prinsippskisse:
# samme 1D-blokk brukes for X, Y og Z

Phi_3D = sp.diag(Phi_1D, Phi_1D, Phi_1D)
Q_3D = sp.diag(Q_1D, Q_1D, Q_1D)

# ============================================================
# OPPGAVE 2c - MÅLEOPPDATERING
# ============================================================
#
# Måleoppdatering:
#
# x_hat = x_tilde + K * (z - H*x_tilde)
#
# Innovasjon:
# z - H*x_tilde
#
# Innovasjonens kovarians:
# S = H*P_tilde*H.T + R
#
# Kalman gain:
# K = P_tilde*H.T*S^-1

# ============================================================
# UTSKRIFT
# ============================================================

print("=" * 80)
print("GMPE240 2024 - OPPGAVE 2 KALMANFILTER")
print("=" * 80)

print("\nOPPGAVE 2a - DYNAMISK MODELL")
print("-" * 80)
print("Modelltype: konstant hastighet-modell")
print("Brukes for: UAV/bevegelse med omtrent konstant hastighet")
print("Tilstand 1D: [posisjon, hastighet]^T")
print("Tilstand 3D: [X, X_dot, Y, Y_dot, Z, Z_dot]^T")
print("F beskriver dynamikken mellom posisjon og hastighet.")
print("G og u beskriver prosesstøy / tilfeldig akselerasjon.")

print("\nF_1D:")
display(F_1D)

print("\nG_1D:")
display(G_1D)

print("\nOPPGAVE 2b - TRANSISJONSMATRISE OG Q")
print("-" * 80)

print("\nPhi_1D:")
display(Phi_1D)

print("\nQ_1D:")
display(Q_1D)

print("\nPhi_3D, blokkdiagonal prinsippskisse:")
display(Phi_3D)

print("\nQ_3D, blokkdiagonal prinsippskisse:")
display(Q_3D)

print("\nOPPGAVE 2c - MÅLEOPPDATERING")
print("-" * 80)

print("Formel for måleoppdatering:")
print("x_hat_k = x_tilde_k + K_k * (z_k - H_k * x_tilde_k)")

print("\nInnovasjon:")
print("z_k - H_k * x_tilde_k")

print("\nInnovasjonens kovarians:")
print("S_k = H_k * P_tilde_k * H_k.T + R_k")

print("\nKalman gain:")
print("K_k = P_tilde_k * H_k.T * inv(S_k)")

print("\nTolkning:")
print("Innovasjonen er forskjellen mellom faktisk måling og forventet måling.")
print("Liten innovasjon betyr at måling og prediksjon stemmer godt.")
print("Stor innovasjon betyr at måling og prediksjon er uenige.")
print("Stor P_tilde betyr usikker prediksjon.")
print("Stor R betyr usikker måling.")
print("Liten R gjør at målingen får mer vekt.")
print("Stor R gjør at filteret stoler mer på prediksjonen.")

GMPE240 2024 - OPPGAVE 2 KALMANFILTER

OPPGAVE 2a - DYNAMISK MODELL
--------------------------------------------------------------------------------
Modelltype: konstant hastighet-modell
Brukes for: UAV/bevegelse med omtrent konstant hastighet
Tilstand 1D: [posisjon, hastighet]^T
Tilstand 3D: [X, X_dot, Y, Y_dot, Z, Z_dot]^T
F beskriver dynamikken mellom posisjon og hastighet.
G og u beskriver prosesstøy / tilfeldig akselerasjon.

F_1D:


Matrix([
[0, 1],
[0, 0]])


G_1D:


Matrix([
[      0],
[sqrt(q)]])


OPPGAVE 2b - TRANSISJONSMATRISE OG Q
--------------------------------------------------------------------------------

Phi_1D:


...


Phi_3D, blokkdiagonal prinsippskisse:


Matrix([
[1, Delta_t, 0,       0, 0,       0],
[0,       1, 0,       0, 0,       0],
[0,       0, 1, Delta_t, 0,       0],
[0,       0, 0,       1, 0,       0],
[0,       0, 0,       0, 1, Delta_t],
[0,       0, 0,       0, 0,       1]])


Q_3D, blokkdiagonal prinsippskisse:


Matrix([
[Delta_t**3*q/3, Delta_t**2*q/2,              0,              0,              0,              0],
[Delta_t**2*q/2,      Delta_t*q,              0,              0,              0,              0],
[             0,              0, Delta_t**3*q/3, Delta_t**2*q/2,              0,              0],
[             0,              0, Delta_t**2*q/2,      Delta_t*q,              0,              0],
[             0,              0,              0,              0, Delta_t**3*q/3, Delta_t**2*q/2],
[             0,              0,              0,              0, Delta_t**2*q/2,      Delta_t*q]])


OPPGAVE 2c - MÅLEOPPDATERING
--------------------------------------------------------------------------------
Formel for måleoppdatering:
x_hat_k = x_tilde_k + K_k * (z_k - H_k * x_tilde_k)

Innovasjon:
z_k - H_k * x_tilde_k

Innovasjonens kovarians:
S_k = H_k * P_tilde_k * H_k.T + R_k

Kalman gain:
K_k = P_tilde_k * H_k.T * inv(S_k)

Tolkning:
Innovasjonen er forskjellen mellom faktisk måling og forventet måling.
Liten innovasjon betyr at måling og prediksjon stemmer godt.
Stor innovasjon betyr at måling og prediksjon er uenige.
Stor P_tilde betyr usikker prediksjon.
Stor R betyr usikker måling.
Liten R gjør at målingen får mer vekt.
Stor R gjør at filteret stoler mer på prediksjonen.
